In [0]:
dbutils.widgets.text("catalog", "ecom_dev")
catalog = dbutils.widgets.get("catalog")

source_dir = f"/Volumes/{catalog}/landing/raw/olist"

In [0]:
TABLES = {
    "customers":                    "olist_customers_dataset.csv",
    "geolocation":                  "olist_geolocation_dataset.csv",
    "orders":                       "olist_orders_dataset.csv",
    "order_items":                  "olist_order_items_dataset.csv",
    "order_payments":               "olist_order_payments_dataset.csv",
    "order_reviews":                "olist_order_reviews_dataset.csv",
    "products":                     "olist_products_dataset.csv",
    "sellers":                      "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

In [0]:
from pyspark.sql import functions as F

def load_to_bronze(table_name, file_name):
    df = (
        spark.read
        .option("header", True)
        .option("multiLine", True)   # review comments contain line breaks
        .option("escape", '"')       # quotes inside text are written as ""
        .csv(f"{source_dir}/{file_name}")
    )

    (
        df
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .write.mode("overwrite")
        .saveAsTable(f"{catalog}.bronze.{table_name}")
    )

    return spark.table(f"{catalog}.bronze.{table_name}").count()

In [0]:
for table_name, file_name in TABLES.items():
    rows = load_to_bronze(table_name, file_name)
    print(f"{table_name:<30} {rows:>10,}")